# Asset Embeddings: Full 80-Quarter Replication

This notebook runs the validated WRDS asset-embeddings pipeline for all 80 quarters from 2005Q1 through 2024Q4. It is the production replication and extension of the quarterly holdings-based analysis.

The validation covers raw S12 and S34 holdings, historical security linkage, historical fund–manager overlap, Jensen-style firm characteristics, paper-consistent sample filters, sparse matrices, RS-Binary and RS-L-Min embeddings, and the relative-valuation benchmark from *Asset Embeddings* by Gabaix, Koijen, Richmond, and Yogo (2025).

All WRDS extracts and derived artifacts are stored locally. Existing valid files are reused, and expensive stages are checkpointed by quarter. Credentials are requested interactively and are never written to disk.


## 1. Imports, project paths, and full-sample configuration

The analysis contains every calendar quarter from 2005Q1 through 2024Q4. Set `FORCE_REBUILD` to `True` only when derived production artifacts must be regenerated. Immutable raw WRDS extracts are never overwritten automatically.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "Asset_Embeddings":
    candidate = PROJECT_ROOT / "Projects" / "Asset_Embeddings"
    if candidate.exists():
        PROJECT_ROOT = candidate.resolve()
if PROJECT_ROOT.name != "Asset_Embeddings":
    raise RuntimeError("Open this notebook from the Asset_Embeddings project directory.")

os.chdir(PROJECT_ROOT)

LOCAL_PACKAGES = PROJECT_ROOT / ".packages"
if str(LOCAL_PACKAGES) not in sys.path:
    sys.path.insert(0, str(LOCAL_PACKAGES))

from scipy import sparse
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


VALIDATION_QUARTERS = [(year, quarter) for year in range(2005, 2025) for quarter in range(1, 5)]
HOLDINGS_PARTITIONS = 16
MINIMUM_DEGREE = 20
MAXIMUM_POSITION_WEIGHT = 0.75
MAXIMUM_EMBEDDING_DIMENSION = 128
RANDOM_SEED = 202451
TEST_FRACTION = 0.20
RIDGE_ALPHAS = np.logspace(-6, 6, 49)
FORCE_REBUILD = False
PIPELINE_VERSION = 2

VALIDATION_ROOT = PROJECT_ROOT / "full_replication_80q"
for folder in ["manifests", "processed", "matrices", "embeddings", "benchmarks", "figures", "logs"]:
    (VALIDATION_ROOT / folder).mkdir(parents=True, exist_ok=True)


def quarter_end(year, quarter):
    return pd.Period(f"{year}Q{quarter}", freq="Q").end_time.normalize()


def quarter_label(year, quarter):
    return f"{year}q{quarter}"


configuration = {
    "quarters": VALIDATION_QUARTERS,
    "holdings_partitions": HOLDINGS_PARTITIONS,
    "minimum_degree": MINIMUM_DEGREE,
    "maximum_position_weight": MAXIMUM_POSITION_WEIGHT,
    "maximum_embedding_dimension": MAXIMUM_EMBEDDING_DIMENSION,
    "random_seed": RANDOM_SEED,
    "test_fraction": TEST_FRACTION,
    "pipeline_version": PIPELINE_VERSION,
}
with open(VALIDATION_ROOT / "manifests" / "configuration.json", "w") as handle:
    json.dump(configuration, handle, indent=2)

print(f"Project root: {PROJECT_ROOT}")
print(f"Analysis quarters: {VALIDATION_QUARTERS}")
print(f"Production output root: {VALIDATION_ROOT}")


## 2. Open or restore the WRDS connection

This cell reuses an active connection when possible. Otherwise, it requests the WRDS username and lets the WRDS client securely prompt for authentication.


In [ ]:
import wrds


def wrds_connection_is_active(connection):
    try:
        connection.raw_sql("select 1 as connection_test")
        return True
    except Exception:
        return False


if "db" in globals() and wrds_connection_is_active(db):
    print("Existing WRDS connection is active.")
else:
    WRDS_USERNAME = os.getenv("WRDS_USERNAME") or input("WRDS username: ").strip()
    if not WRDS_USERNAME:
        raise ValueError("A WRDS username is required.")
    db = wrds.Connection(wrds_username=WRDS_USERNAME)
    print("WRDS connection established.")


## 3. Checkpoint and data-normalization utilities

These helpers standardize identifiers, save Parquet files, validate checkpoints, and avoid repeating completed work.


In [ ]:
def normalize_cusip(series):
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.replace(r"[^0-9A-Z]", "", regex=True)
        .str[:8]
    )


def parquet_is_valid(path, required_columns=None):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        frame = pd.read_parquet(path)
    except Exception:
        return False
    if required_columns and not set(required_columns).issubset(frame.columns):
        return False
    return True


def save_parquet(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(path, index=False)


def market_equity_weighted_average(group, variable, weight="me"):
    valid = group[variable].notna() & group[weight].notna() & group[weight].gt(0)
    if not valid.any():
        return np.nan
    return float(np.average(group.loc[valid, variable], weights=group.loc[valid, weight]))


def recompute_portfolio_fields(frame):
    frame = frame.copy()
    frame["portfolio_value"] = frame.groupby("investor_key")["dollar_holding"].transform("sum")
    frame["portfolio_weight"] = frame["dollar_holding"] / frame["portfolio_value"]
    frame["log_dollar_holding"] = np.log(frame["dollar_holding"])
    frame["firm_rank"] = frame.groupby("investor_key")["dollar_holding"].rank(method="first", ascending=False).astype("int32")
    return frame


def iterative_degree_filter(frame, minimum_degree=20):
    current = frame.copy()
    audit = []
    iteration = 0
    while True:
        iteration += 1
        rows_before = len(current)
        investor_degree = current.groupby("investor_key")["permco"].nunique()
        firm_degree = current.groupby("permco")["investor_key"].nunique()
        current = current[
            current["investor_key"].isin(investor_degree[investor_degree >= minimum_degree].index)
            & current["permco"].isin(firm_degree[firm_degree >= minimum_degree].index)
        ].copy()
        audit.append({
            "iteration": iteration,
            "rows_before": rows_before,
            "rows_after": len(current),
            "investors_after": current["investor_key"].nunique(),
            "firms_after": current["permco"].nunique(),
        })
        if len(current) == rows_before:
            break
        if current.empty:
            raise ValueError("Degree filtering removed the entire sample.")
    return current, pd.DataFrame(audit)


## 4. Historical point-in-time CRSP security crosswalk

For each validation quarter, the active CRSP name interval maps the reported eight-character CUSIP to PERMNO and PERMCO. This prevents the 2024 crosswalk from being applied retrospectively.


In [ ]:
def obtain_crsp_crosswalk(year, quarter):
    date = quarter_end(year, quarter)
    label = quarter_label(year, quarter)
    path = PROJECT_ROOT / "data" / "crosswalks" / "crsp" / f"year={year}" / f"quarter={quarter}" / f"stocknames_{label}.parquet"
    required = ["ncusip", "permno", "permco", "shrcd"]
    if parquet_is_valid(path, required) and not FORCE_REBUILD:
        frame = pd.read_parquet(path)
    else:
        query = f"""
            select ncusip, permno, permco, namedt, nameenddt,
                   shrcd, exchcd, comnam, ticker
            from crsp_a_stock.stocknames
            where namedt <= '{date:%Y-%m-%d}'
              and (nameenddt >= '{date:%Y-%m-%d}' or nameenddt is null)
        """
        frame = db.raw_sql(query, date_cols=["namedt", "nameenddt"])
        frame["ncusip"] = normalize_cusip(frame["ncusip"])
        for column in ["permno", "permco", "shrcd", "exchcd"]:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frame = frame.dropna(subset=["ncusip", "permno", "permco"]).copy()
        counts = frame.groupby("ncusip")["permno"].nunique()
        ambiguous = set(counts[counts > 1].index)
        if ambiguous:
            frame = frame[~frame["ncusip"].isin(ambiguous)].copy()
        save_parquet(frame, path)
    return frame


crosswalk_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    crosswalk = obtain_crsp_crosswalk(year, quarter)
    crosswalk_inventory.append({
        "year": year, "quarter": quarter, "rows": len(crosswalk),
        "permnos": crosswalk["permno"].nunique(), "permcos": crosswalk["permco"].nunique(),
        "ordinary_common_rows": int(crosswalk["shrcd"].isin([10, 11]).sum()),
    })
display(pd.DataFrame(crosswalk_inventory))


## 5. Partitioned S12 and S34 downloads

Holdings are pulled from the dedicated `tr_mutualfunds.s12` and `tr_13f.s34` tables. The baseline uses strict quarter-end observations with `fdate = rdate = quarter end`. Sixteen investor-ID partitions provide restartable downloads and bounded memory use.


In [ ]:
def holdings_specification(source):
    if source == "s12":
        return "tr_mutualfunds.s12", "fundno"
    if source == "s34":
        return "tr_13f.s34", "mgrno"
    raise ValueError(source)


def obtain_raw_holdings(source, year, quarter):
    table, investor_column = holdings_specification(source)
    date = quarter_end(year, quarter)
    directory = PROJECT_ROOT / "data" / "raw" / source / f"year={year}" / f"quarter={quarter}"
    directory.mkdir(parents=True, exist_ok=True)
    audit_rows = []
    for partition in range(HOLDINGS_PARTITIONS):
        path = directory / f"part-{partition:02d}-of-{HOLDINGS_PARTITIONS:02d}.parquet"
        start = time.perf_counter()
        if parquet_is_valid(path, ["fdate", "rdate", investor_column, "cusip", "shares", "prc"]):
            frame = pd.read_parquet(path)
            status = "checkpoint"
        else:
            query = f"""
                select *
                from {table}
                where fdate = '{date:%Y-%m-%d}'
                  and rdate = '{date:%Y-%m-%d}'
                  and mod(cast({investor_column} as bigint), {HOLDINGS_PARTITIONS}) = {partition}
            """
            frame = db.raw_sql(query, date_cols=["fdate", "rdate", "prdate"])
            save_parquet(frame, path)
            status = "downloaded"
        audit_rows.append({
            "source": source, "year": year, "quarter": quarter, "partition": partition,
            "status": status, "rows": len(frame),
            "investors": frame[investor_column].nunique(),
            "securities": frame["cusip"].nunique(),
            "seconds": time.perf_counter() - start,
            "file_mb": path.stat().st_size / (1024 ** 2),
        })
    return directory, pd.DataFrame(audit_rows)


download_audits = []
for year, quarter in VALIDATION_QUARTERS:
    for source in ["s12", "s34"]:
        _, audit = obtain_raw_holdings(source, year, quarter)
        download_audits.append(audit)
download_audit = pd.concat(download_audits, ignore_index=True)
save_parquet(download_audit, VALIDATION_ROOT / "manifests" / "holdings_download_audit.parquet")
display(download_audit.groupby(["year", "quarter", "source"], as_index=False).agg(rows=("rows", "sum"), file_mb=("file_mb", "sum"), downloaded=("status", lambda x: int((x == "downloaded").sum()))))


## 6. Historical S12 fund-to-S34 manager mappings

The `s12type5` crosswalk is downloaded separately for each quarter. An S34 parent manager is removed from the combined sample only when it is linked to an active S12 fund in that same quarter.


In [ ]:
def obtain_s12_s34_mapping(year, quarter):
    date = quarter_end(year, quarter)
    label = quarter_label(year, quarter)
    directory = PROJECT_ROOT / "data" / "crosswalks" / "s12_s34" / f"year={year}" / f"quarter={quarter}"
    path = directory / f"s12type5_{label}.parquet"
    if parquet_is_valid(path, ["fdate", "fundno", "mgrcocd"]):
        frame = pd.read_parquet(path)
    else:
        legacy = PROJECT_ROOT / "data" / "crosswalks" / "s12_s34" / f"s12type5_{label}_raw.parquet"
        if parquet_is_valid(legacy, ["fdate", "fundno", "mgrcocd"]):
            frame = pd.read_parquet(legacy)
        else:
            query = f"""
                select fdate, fundno, fundname, mgrco, mgrcocd
                from tr_mutualfunds.s12type5
                where fdate = '{date:%Y-%m-%d}'
            """
            frame = db.raw_sql(query, date_cols=["fdate"])
            save_parquet(frame, path)

    # WRDS has no s12type5 observations for 2011Q3. For any such gap,
    # use only the most recent mapping available before the quarter,
    # avoiding future information and preserving point-in-time logic.
    if frame.empty:
        resolved_path = VALIDATION_ROOT / "processed" / label / "s12_s34_mapping_resolved.parquet"
        if parquet_is_valid(resolved_path, ["fdate", "fundno", "mgrcocd"]):
            frame = pd.read_parquet(resolved_path)
        else:
            fallback_query = f"""
                select fdate, fundno, fundname, mgrco, mgrcocd
                from tr_mutualfunds.s12type5
                where fdate = (
                    select max(fdate)
                    from tr_mutualfunds.s12type5
                    where fdate < '{date:%Y-%m-%d}'
                )
            """
            frame = db.raw_sql(fallback_query, date_cols=["fdate"])
            if frame.empty:
                raise ValueError(f"No contemporaneous or prior S12-S34 mapping exists for {label}.")
            save_parquet(frame, resolved_path)
        frame = frame.copy()
        frame["mapping_rule"] = "latest_prior_available_fdate"
    else:
        frame = frame.copy()
        frame["mapping_rule"] = "exact_quarter_fdate"
    return frame


mapping_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    mapping = obtain_s12_s34_mapping(year, quarter)
    mapping_inventory.append({"year": year, "quarter": quarter, "rows": len(mapping), "funds": mapping["fundno"].nunique(), "managers": mapping["mgrcocd"].nunique()})
display(pd.DataFrame(mapping_inventory))


## 7. Jensen-style firm characteristics and exact size categories

The contributed `global_factor` table supplies the Appendix D.1.2 variables: `beta_60m`, `at_gr1`, `gp_at`, and `div_at`, plus book and market equity. Security observations are aggregated to PERMCO using market-equity weights. Firms must have at least one small, large, or mega security; nano and micro firms are excluded.


In [ ]:
def obtain_jkp_security_data(year, quarter):
    date = quarter_end(year, quarter)
    label = quarter_label(year, quarter)
    directory = PROJECT_ROOT / "data" / "raw" / "jkp_characteristics" / f"year={year}" / f"quarter={quarter}"
    path = directory / f"global_factor_paper_exact_{label}.parquet"
    required = ["permno", "permco", "me", "book_equity", "beta_60m", "at_gr1", "gp_at", "div_at"]
    if parquet_is_valid(path, required):
        return pd.read_parquet(path)
    query = f"""
        select id, permno, permco, gvkey, iid, excntry, common, primary_sec,
               obs_main, exch_main, crsp_shrcd, crsp_exchcd, date, eom,
               size_grp, me, me_company, book_equity,
               beta_60m, at_gr1, gp_at, div_at
        from contrib_global_factor.global_factor
        where eom = '{date:%Y-%m-%d}' and excntry = 'USA'
    """
    frame = db.raw_sql(query, date_cols=["date", "eom"])
    save_parquet(frame, path)
    return frame


def construct_jkp_firm_panel(year, quarter):
    label = quarter_label(year, quarter)
    output = VALIDATION_ROOT / "processed" / label / "jkp_firm_panel.parquet"
    if parquet_is_valid(output, ["permco", "beta_60m_paper", "at_gr1_paper", "gp_at_paper", "div_at_paper"]) and not FORCE_REBUILD:
        return pd.read_parquet(output)
    frame = obtain_jkp_security_data(year, quarter).copy()
    numeric = ["permno", "permco", "common", "primary_sec", "obs_main", "exch_main", "crsp_shrcd", "me", "me_company", "book_equity", "beta_60m", "at_gr1", "gp_at", "div_at"]
    for column in numeric:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.dropna(subset=["permno", "permco"]).copy()
    frame["permno"] = frame["permno"].astype("int64")
    frame["permco"] = frame["permco"].astype("int64")
    frame["size_group_clean"] = frame["size_grp"].astype("string").str.lower().str.strip()
    frame = frame[frame["common"].eq(1) | frame["crsp_shrcd"].isin([10, 11])].copy()
    frame["eligible_size"] = frame["size_group_clean"].isin(["small", "large", "mega"])
    rows = []
    for permco, group in frame.groupby("permco", sort=True):
        positive_me = group.loc[group["me"].gt(0), "me"]
        positive_be = group.loc[group["book_equity"].gt(0), "book_equity"]
        row = {
            "permco": int(permco),
            "eligible_size_security": bool(group["eligible_size"].any()),
            "underlying_permnos": int(group["permno"].nunique()),
            "jkp_firm_market_equity": float(positive_me.sum()) if len(positive_me) else np.nan,
            "jkp_book_equity": float(positive_be.max()) if len(positive_be) else np.nan,
        }
        for variable in ["beta_60m", "at_gr1", "gp_at", "div_at"]:
            row[variable] = market_equity_weighted_average(group, variable)
        rows.append(row)
    firms = pd.DataFrame(rows)
    firms = firms[firms["eligible_size_security"] & firms["jkp_book_equity"].gt(0)].copy()
    firms = firms.dropna(subset=["beta_60m", "at_gr1", "gp_at", "div_at"]).copy()
    firms["beta_60m_paper"] = firms["beta_60m"]
    for variable in ["at_gr1", "gp_at"]:
        lower, upper = firms[variable].quantile([0.01, 0.99])
        firms[f"{variable}_paper"] = firms[variable].clip(lower=lower, upper=upper)
    firms["div_at_paper"] = firms["div_at"].clip(upper=firms["div_at"].quantile(0.99))
    save_parquet(firms, output)
    return firms


jkp_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    firms = construct_jkp_firm_panel(year, quarter)
    jkp_inventory.append({"year": year, "quarter": quarter, "complete_case_firms": len(firms), "eligible_permcos": firms["permco"].nunique()})
display(pd.DataFrame(jkp_inventory))


## 8. Link holdings, aggregate to firms, and construct the combined sample

CUSIPs are matched using the quarter-specific CRSP crosswalk. S12 shares are actual share counts, so dollar holdings equal shares times absolute price. Securities are aggregated to PERMCO. The combined sample removes overlapping S34 parents, excludes portfolios with a largest position above 75%, and iteratively requires at least 20 firms per investor and 20 investors per firm.


In [ ]:
def link_and_aggregate_source(source, year, quarter):
    label = quarter_label(year, quarter)
    output = VALIDATION_ROOT / "processed" / label / f"{source}_firm_holdings_prefilter.parquet"
    if parquet_is_valid(output, ["investor_key", "permco", "dollar_holding"]) and not FORCE_REBUILD:
        return pd.read_parquet(output)
    raw_directory, _ = obtain_raw_holdings(source, year, quarter)
    crosswalk = obtain_crsp_crosswalk(year, quarter).copy()
    crosswalk["ncusip"] = normalize_cusip(crosswalk["ncusip"])
    crosswalk = crosswalk[crosswalk["shrcd"].isin([10, 11])].copy()
    crosswalk = crosswalk.drop_duplicates("ncusip")
    investor_column = "fundno" if source == "s12" else "mgrno"
    name_column = "fundname" if source == "s12" else "mgrname"
    pieces = []
    for path in sorted(raw_directory.glob("part-*-of-*.parquet")):
        raw = pd.read_parquet(path)
        raw["cusip8"] = normalize_cusip(raw["cusip"])
        raw["shares"] = pd.to_numeric(raw["shares"], errors="coerce")
        raw["prc"] = pd.to_numeric(raw["prc"], errors="coerce")
        raw[investor_column] = pd.to_numeric(raw[investor_column], errors="coerce")
        linked = raw.merge(crosswalk[["ncusip", "permno", "permco", "comnam"]], left_on="cusip8", right_on="ncusip", how="inner", validate="many_to_one")
        linked["dollar_holding"] = linked["shares"] * linked["prc"].abs()
        linked = linked[linked["dollar_holding"].gt(0) & linked[investor_column].notna()].copy()
        linked["investor_id"] = linked[investor_column].astype("int64")
        linked["investor_key"] = source + ":" + linked["investor_id"].astype(str)
        linked["investor_name"] = linked[name_column].astype("string")
        linked["permco"] = linked["permco"].astype("int64")
        pieces.append(linked[["fdate", "rdate", "investor_id", "investor_key", "investor_name", "permco", "comnam", "permno", "dollar_holding"]])
    security = pd.concat(pieces, ignore_index=True)
    firm = security.groupby(["fdate", "rdate", "investor_id", "investor_key", "investor_name", "permco"], as_index=False, dropna=False).agg(
        firm_name=("comnam", "first"), dollar_holding=("dollar_holding", "sum"),
        underlying_security_rows=("permno", "size"), underlying_permnos=("permno", "nunique"),
    )
    firm.insert(0, "source", source)
    save_parquet(firm, output)
    return firm


def construct_final_combined_holdings(year, quarter):
    label = quarter_label(year, quarter)
    output = VALIDATION_ROOT / "processed" / label / "combined_final_holdings.parquet"
    audit_path = VALIDATION_ROOT / "processed" / label / "filter_audit.parquet"
    if parquet_is_valid(output, ["investor_key", "permco", "portfolio_weight", "pipeline_version"]) and not FORCE_REBUILD:
        return pd.read_parquet(output), pd.read_parquet(audit_path)
    s12 = link_and_aggregate_source("s12", year, quarter)
    s34 = link_and_aggregate_source("s34", year, quarter)
    mapping = obtain_s12_s34_mapping(year, quarter).copy()
    mapping["fundno"] = pd.to_numeric(mapping["fundno"], errors="coerce")
    mapping["mgrcocd"] = pd.to_numeric(mapping["mgrcocd"], errors="coerce")
    raw_s12_directory, _ = obtain_raw_holdings("s12", year, quarter)
    active_s12 = set()
    for raw_path in sorted(raw_s12_directory.glob("part-*-of-*.parquet")):
        raw_fund_ids = pd.read_parquet(raw_path, columns=["fundno"])["fundno"]
        active_s12.update(pd.to_numeric(raw_fund_ids, errors="coerce").dropna().astype("int64"))
    active_s34 = set(s34["investor_id"])
    overlapping_managers = set(mapping.loc[mapping["fundno"].isin(active_s12), "mgrcocd"].dropna().astype("int64")) & active_s34
    s34 = s34[~s34["investor_id"].isin(overlapping_managers)].copy()
    combined = pd.concat([s12, s34], ignore_index=True)
    combined["pipeline_version"] = PIPELINE_VERSION
    eligible_firms = construct_jkp_firm_panel(year, quarter)[["permco", "jkp_firm_market_equity"]]
    combined = combined.merge(eligible_firms, on="permco", how="inner", validate="many_to_one")
    combined = recompute_portfolio_fields(combined)
    concentration = combined.groupby("investor_key")["portfolio_weight"].max()
    concentrated = set(concentration[concentration > MAXIMUM_POSITION_WEIGHT].index)
    combined = combined[~combined["investor_key"].isin(concentrated)].copy()
    combined, iteration_audit = iterative_degree_filter(combined, MINIMUM_DEGREE)
    combined = recompute_portfolio_fields(combined)
    audit = pd.DataFrame({
        "year": [year], "quarter": [quarter], "active_s34_managers_removed": [len(overlapping_managers)],
        "concentrated_investors_removed": [len(concentrated)], "final_rows": [len(combined)],
        "final_investors": [combined["investor_key"].nunique()], "final_firms": [combined["permco"].nunique()],
        "degree_iterations": [len(iteration_audit)], "pipeline_version": [PIPELINE_VERSION],
        "s34_overlap_rule": ["mapped_to_any_raw_active_s12_fund"],
    })
    save_parquet(combined, output)
    save_parquet(audit, audit_path)
    save_parquet(iteration_audit, VALIDATION_ROOT / "processed" / label / "degree_iteration_audit.parquet")
    return combined, audit


filter_audits = []
for year, quarter in VALIDATION_QUARTERS:
    holdings, audit = construct_final_combined_holdings(year, quarter)
    filter_audits.append(audit)
filter_audit = pd.concat(filter_audits, ignore_index=True)
save_parquet(filter_audit, VALIDATION_ROOT / "manifests" / "quarterly_filter_audit.parquet")
display(filter_audit)


## 9. Sparse holdings matrices

The final combined holdings become aligned investor-by-firm matrices for binary ownership and positive log dollar holdings. Mapping tables make every row and column recoverable.


In [ ]:
def construct_sparse_matrices(year, quarter):
    label = quarter_label(year, quarter)
    directory = VALIDATION_ROOT / "matrices" / label
    binary_path = directory / "binary.npz"
    log_path = directory / "log_dollar_holdings_positive.npz"
    investor_path = directory / "investor_mapping.parquet"
    firm_path = directory / "firm_mapping.parquet"
    summary_path = directory / "matrix_summary.parquet"
    if binary_path.exists() and log_path.exists() and parquet_is_valid(investor_path) and parquet_is_valid(firm_path) and parquet_is_valid(summary_path, ["pipeline_version"]) and not FORCE_REBUILD:
        return sparse.load_npz(binary_path), sparse.load_npz(log_path), pd.read_parquet(investor_path), pd.read_parquet(firm_path)
    holdings, _ = construct_final_combined_holdings(year, quarter)
    directory.mkdir(parents=True, exist_ok=True)
    investor_mapping = holdings[["investor_key", "source", "investor_id", "investor_name"]].drop_duplicates("investor_key").sort_values("investor_key").reset_index(drop=True)
    investor_mapping["investor_index"] = np.arange(len(investor_mapping), dtype="int32")
    firm_mapping = holdings[["permco", "firm_name"]].drop_duplicates("permco").sort_values("permco").reset_index(drop=True)
    firm_mapping["firm_index"] = np.arange(len(firm_mapping), dtype="int32")
    indexed = holdings.merge(investor_mapping[["investor_key", "investor_index"]], on="investor_key", validate="many_to_one").merge(firm_mapping[["permco", "firm_index"]], on="permco", validate="many_to_one")
    rows = indexed["investor_index"].to_numpy()
    cols = indexed["firm_index"].to_numpy()
    shape = (len(investor_mapping), len(firm_mapping))
    binary = sparse.csr_matrix((np.ones(len(indexed), dtype="float32"), (rows, cols)), shape=shape)
    log_values = np.log(indexed["dollar_holding"].to_numpy(dtype="float64")).astype("float32")
    log_matrix = sparse.csr_matrix((log_values, (rows, cols)), shape=shape)
    sparse.save_npz(binary_path, binary, compressed=True)
    sparse.save_npz(log_path, log_matrix, compressed=True)
    save_parquet(investor_mapping, investor_path)
    save_parquet(firm_mapping, firm_path)
    summary = pd.DataFrame({"investors": [shape[0]], "firms": [shape[1]], "nonzero_holdings": [binary.nnz], "density": [binary.nnz / (shape[0] * shape[1])], "pipeline_version": [PIPELINE_VERSION]})
    save_parquet(summary, summary_path)
    return binary, log_matrix, investor_mapping, firm_mapping


matrix_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    binary, _, investors, firms = construct_sparse_matrices(year, quarter)
    matrix_inventory.append({"year": year, "quarter": quarter, "investors": len(investors), "firms": len(firms), "nonzero": binary.nnz, "density": binary.nnz / np.prod(binary.shape)})
display(pd.DataFrame(matrix_inventory))


## 10. RS-Binary and RS-L-Min embeddings

RS-Binary applies centered PCA to the binary holdings matrix. RS-L-Min removes investor and firm fixed effects from positive log holdings, replaces each missing position with that investor's minimum observed residual, and applies centered PCA. Asset embeddings are normalized so their cross-sectional Gram matrix is approximately the identity.


In [ ]:
def normalized_asset_embeddings(pca, number_of_firms):
    return (pca.components_.T * np.sqrt(number_of_firms)).astype("float32")


def fit_embeddings(year, quarter):
    label = quarter_label(year, quarter)
    binary, log_sparse, investor_mapping, firm_mapping = construct_sparse_matrices(year, quarter)
    number_of_components = min(MAXIMUM_EMBEDDING_DIMENSION, binary.shape[0] - 1, binary.shape[1] - 1)
    outputs = {}
    for model in ["rs_binary", "rs_l_min"]:
        directory = VALIDATION_ROOT / "embeddings" / label / model
        path = directory / "asset_embeddings_normalized_labeled.parquet"
        fit_summary_path = directory / "fit_summary.parquet"
        if parquet_is_valid(path, ["permco", "embedding_001"]) and parquet_is_valid(fit_summary_path, ["pipeline_version"]) and not FORCE_REBUILD:
            outputs[model] = pd.read_parquet(path)
            continue
        directory.mkdir(parents=True, exist_ok=True)
        if model == "rs_binary":
            dense = binary.toarray().astype("float32")
        else:
            coo = log_sparse.tocoo()
            observed_counts_i = np.bincount(coo.row, minlength=log_sparse.shape[0])
            investor_sums = np.bincount(coo.row, weights=coo.data, minlength=log_sparse.shape[0])
            investor_fe = investor_sums / observed_counts_i
            investor_demeaned = coo.data - investor_fe[coo.row]
            observed_counts_a = np.bincount(coo.col, minlength=log_sparse.shape[1])
            firm_sums = np.bincount(coo.col, weights=investor_demeaned, minlength=log_sparse.shape[1])
            firm_fe = firm_sums / observed_counts_a
            residual = investor_demeaned - firm_fe[coo.col]
            investor_minimum = np.full(log_sparse.shape[0], np.inf, dtype="float64")
            np.minimum.at(investor_minimum, coo.row, residual)
            dense = np.repeat(investor_minimum[:, None], log_sparse.shape[1], axis=1).astype("float32")
            dense[coo.row, coo.col] = residual.astype("float32")
        start = time.perf_counter()
        pca = PCA(n_components=number_of_components, svd_solver="randomized", random_state=RANDOM_SEED)
        pca.fit(dense)
        asset = normalized_asset_embeddings(pca, dense.shape[1])
        labeled = firm_mapping[["permco", "firm_name", "firm_index"]].copy()
        for component in range(number_of_components):
            labeled[f"embedding_{component + 1:03d}"] = asset[:, component]
        save_parquet(labeled, path)
        fit_summary = pd.DataFrame({
            "model": [model], "investors": [dense.shape[0]], "firms": [dense.shape[1]],
            "components": [number_of_components], "fit_seconds": [time.perf_counter() - start],
            "cumulative_explained_variance_128": [float(pca.explained_variance_ratio_.sum())],
            "pipeline_version": [PIPELINE_VERSION],
        })
        save_parquet(fit_summary, fit_summary_path)
        np.save(directory / "explained_variance_ratio.npy", pca.explained_variance_ratio_)
        outputs[model] = labeled
        del dense
    return outputs


embedding_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    outputs = fit_embeddings(year, quarter)
    for model, frame in outputs.items():
        embedding_inventory.append({"year": year, "quarter": quarter, "model": model, "firms": len(frame), "dimensions": len([c for c in frame if c.startswith("embedding_")])})
display(pd.DataFrame(embedding_inventory))


## 11. Paper-style relative-valuation benchmark by quarter

Within the final common sample, log market equity is regressed on log book equity and the residual becomes relative valuation. A fixed 80/20 firm split is shared by Gaussian placebos, Beta, four characteristics, RS-Binary, RS-L-Min, and embedding-plus-characteristic models. Ridge penalties are chosen by ten-fold cross-validation. The paper's variance-based test R-squared is calculated separately in every quarter.


In [ ]:
def run_valuation_benchmark(year, quarter):
    label = quarter_label(year, quarter)
    directory = VALIDATION_ROOT / "benchmarks" / label
    result_path = directory / "relative_valuation_results.parquet"
    if parquet_is_valid(result_path, ["model", "test_variance_r2", "pipeline_version"]) and not FORCE_REBUILD:
        return pd.read_parquet(result_path)
    directory.mkdir(parents=True, exist_ok=True)
    characteristics = construct_jkp_firm_panel(year, quarter).copy()
    embeddings = fit_embeddings(year, quarter)
    panel = characteristics.copy()
    for model, frame in embeddings.items():
        columns = sorted(c for c in frame if c.startswith("embedding_"))
        renamed = frame[["permco"] + columns].rename(columns={c: f"{model}_{c}" for c in columns})
        panel = panel.merge(renamed, on="permco", how="inner", validate="one_to_one")
    panel = panel[panel["jkp_firm_market_equity"].gt(0) & panel["jkp_book_equity"].gt(0)].sort_values("permco").reset_index(drop=True)
    panel["log_me"] = np.log(panel["jkp_firm_market_equity"])
    panel["log_be"] = np.log(panel["jkp_book_equity"])
    valuation = LinearRegression().fit(panel[["log_be"]], panel["log_me"])
    panel["relative_valuation"] = panel["log_me"] - valuation.predict(panel[["log_be"]])
    n = len(panel)
    rng = np.random.default_rng(RANDOM_SEED + year * 10 + quarter)
    order = rng.permutation(n)
    n_test = int(np.ceil(TEST_FRACTION * n))
    test_positions, train_positions = order[:n_test], order[n_test:]
    placebo = rng.standard_normal((n, 10))
    for index in range(10):
        panel[f"placebo_{index + 1:03d}"] = placebo[:, index]
    characteristics_columns = ["beta_60m_paper", "at_gr1_paper", "gp_at_paper", "div_at_paper"]
    binary = [f"rs_binary_embedding_{i:03d}" for i in range(1, 11)]
    lmin = [f"rs_l_min_embedding_{i:03d}" for i in range(1, 11)]
    placebo_columns = [f"placebo_{i:03d}" for i in range(1, 11)]
    models = {
        "placebo_4": placebo_columns[:4], "placebo_10": placebo_columns,
        "beta": ["beta_60m_paper"], "characteristics_4": characteristics_columns,
        "rs_binary_4": binary[:4], "rs_binary_10": binary,
        "rs_l_min_4": lmin[:4], "rs_l_min_10": lmin,
        "rs_binary_10_plus_characteristics": binary + characteristics_columns,
        "rs_l_min_10_plus_characteristics": lmin + characteristics_columns,
    }
    y = panel["relative_valuation"].to_numpy(dtype="float64")
    y_train, y_test = y[train_positions], y[test_positions]
    cv = KFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
    rows = []
    predictions = []
    for model, columns in models.items():
        X = panel[columns].to_numpy(dtype="float64")
        scaler = StandardScaler().fit(X[train_positions])
        X_train, X_test = scaler.transform(X[train_positions]), scaler.transform(X[test_positions])
        ridge = RidgeCV(alphas=RIDGE_ALPHAS, cv=cv, scoring="neg_mean_squared_error").fit(X_train, y_train)
        prediction = ridge.predict(X_test)
        residual = y_test - prediction
        rows.append({
            "year": year, "quarter": quarter, "model": model, "predictors": len(columns),
            "firms": n, "training_firms": len(train_positions), "test_firms": len(test_positions),
            "selected_alpha": float(ridge.alpha_),
            "test_variance_r2": float(1 - np.var(residual) / np.var(y_test)),
            "test_conventional_r2": float(r2_score(y_test, prediction)),
            "test_prediction_correlation": float(np.corrcoef(y_test, prediction)[0, 1]),
            "book_equity_slope": float(valuation.coef_[0]),
            "pipeline_version": PIPELINE_VERSION,
        })
        prediction_frame = panel.iloc[test_positions][["permco"]].copy()
        prediction_frame["model"] = model
        prediction_frame["actual"] = y_test
        prediction_frame["predicted"] = prediction
        predictions.append(prediction_frame)
    results = pd.DataFrame(rows)
    panel["sample"] = "train"
    panel.loc[test_positions, "sample"] = "test"
    save_parquet(results, result_path)
    save_parquet(pd.concat(predictions, ignore_index=True), directory / "test_predictions.parquet")
    save_parquet(panel, directory / "evaluation_sample_and_split.parquet")
    return results


quarterly_results = []
for year, quarter in VALIDATION_QUARTERS:
    print(f"Running valuation benchmark for {year}Q{quarter}...")
    quarterly_results.append(run_valuation_benchmark(year, quarter))
quarterly_results = pd.concat(quarterly_results, ignore_index=True)
save_parquet(quarterly_results, VALIDATION_ROOT / "benchmarks" / "eighty_quarter_results.parquet")
display(quarterly_results.pivot(index="model", columns=["year", "quarter"], values="test_variance_r2"))


## 11.1. Automatically repair missing historical manager mappings

The WRDS `s12type5` table has an empty cross section in 2011Q3. An empty mapping would retain S34 parent managers whose active S12 funds are already represented, creating duplicate institutional exposure.

For an empty exact-quarter mapping only, the pipeline carries forward the latest mapping dated strictly before the quarter. This is a point-in-time fallback and does not introduce future information. The repair is targeted: it rebuilds only affected quarters and their downstream matrices, embeddings, and benchmarks. All other quarterly checkpoints remain unchanged.


In [ ]:
mapping_gap_quarters = []
for year, quarter in VALIDATION_QUARTERS:
    label = quarter_label(year, quarter)
    exact_mapping_path = (
        PROJECT_ROOT / "data" / "crosswalks" / "s12_s34"
        / f"year={year}" / f"quarter={quarter}"
        / f"s12type5_{label}.parquet"
    )
    if exact_mapping_path.exists() and pd.read_parquet(exact_mapping_path).empty:
        mapping_gap_quarters.append((year, quarter))

if mapping_gap_quarters:
    print(f"Repairing mapping gaps: {mapping_gap_quarters}")
    previous_force_rebuild = FORCE_REBUILD
    FORCE_REBUILD = True
    repaired_rows = []
    for year, quarter in mapping_gap_quarters:
        resolved_mapping = obtain_s12_s34_mapping(year, quarter)
        holdings, audit = construct_final_combined_holdings(year, quarter)
        construct_sparse_matrices(year, quarter)
        fit_embeddings(year, quarter)
        result = run_valuation_benchmark(year, quarter)
        repaired_rows.append({
            "year": year,
            "quarter": quarter,
            "mapping_source_fdate": resolved_mapping["fdate"].max(),
            "mapping_rule": resolved_mapping["mapping_rule"].iloc[0],
            "s34_managers_removed": int(audit["active_s34_managers_removed"].iloc[0]),
            "final_investors": int(audit["final_investors"].iloc[0]),
            "final_firms": int(audit["final_firms"].iloc[0]),
        })
    FORCE_REBUILD = previous_force_rebuild
    mapping_gap_audit = pd.DataFrame(repaired_rows)
    save_parquet(mapping_gap_audit, VALIDATION_ROOT / "manifests" / "mapping_gap_repair_audit.parquet")

    # Refresh the aggregate audit and benchmark objects so Section 12
    # automatically uses the repaired quarter.
    filter_audit = pd.concat(
        [
            pd.read_parquet(VALIDATION_ROOT / "processed" / quarter_label(year, quarter) / "filter_audit.parquet")
            for year, quarter in VALIDATION_QUARTERS
        ],
        ignore_index=True,
    )
    save_parquet(filter_audit, VALIDATION_ROOT / "manifests" / "quarterly_filter_audit.parquet")
    quarterly_results = pd.concat(
        [
            pd.read_parquet(VALIDATION_ROOT / "benchmarks" / quarter_label(year, quarter) / "relative_valuation_results.parquet")
            for year, quarter in VALIDATION_QUARTERS
        ],
        ignore_index=True,
    )
    save_parquet(quarterly_results, VALIDATION_ROOT / "benchmarks" / "eighty_quarter_results.parquet")
    display(mapping_gap_audit)
else:
    print("No empty exact-quarter S12-S34 mappings were found.")


## 12. Full-sample quarterly summary and visualization

The paper averages quarterly variance ratios rather than pooling observations. This section reports the mean and dispersion across all 80 quarters, stores the production estimates, and visualizes the complete time series and full-sample averages.


In [ ]:
summary = quarterly_results.groupby("model", as_index=False).agg(
    quarters=("quarter", "size"), mean_test_variance_r2=("test_variance_r2", "mean"),
    minimum_test_variance_r2=("test_variance_r2", "min"), maximum_test_variance_r2=("test_variance_r2", "max"),
    standard_deviation_test_r2=("test_variance_r2", "std"), mean_firms=("firms", "mean"),
).sort_values("mean_test_variance_r2", ascending=False).reset_index(drop=True)
save_parquet(summary, VALIDATION_ROOT / "benchmarks" / "eighty_quarter_summary.parquet")
display(summary)

plot_models = ["placebo_4", "beta", "characteristics_4", "rs_binary_4", "rs_l_min_4", "rs_binary_10", "rs_l_min_10"]
plot_labels = {
    "placebo_4": "Placebo 4D",
    "beta": "Beta",
    "characteristics_4": "Four chars.",
    "rs_binary_4": "RS-Binary 4D",
    "rs_l_min_4": "RS-L-Min 4D",
    "rs_binary_10": "RS-Binary 10D",
    "rs_l_min_10": "RS-L-Min 10D",
}
plot_data = quarterly_results[quarterly_results["model"].isin(plot_models)].copy()
plot_data["quarter_date"] = pd.PeriodIndex(
    [f"{year}Q{quarter}" for year, quarter in zip(plot_data["year"], plot_data["quarter"])],
    freq="Q",
).to_timestamp(how="end").normalize()

figure, axes = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={"width_ratios": [2.2, 1.0], "wspace": 0.25})
time_axis, average_axis = axes

colors = {
    "placebo_4": "#AEB4BE",
    "beta": "#C9A227",
    "characteristics_4": "#D9822B",
    "rs_binary_4": "#75AADB",
    "rs_l_min_4": "#76B58A",
    "rs_binary_10": "#286A9D",
    "rs_l_min_10": "#337A4D",
}

for model in plot_models:
    model_data = plot_data[plot_data["model"] == model].sort_values("quarter_date")
    time_axis.plot(
        model_data["quarter_date"],
        model_data["test_variance_r2"],
        label=plot_labels[model],
        color=colors[model],
        linewidth=2.0 if "10" in model else 1.35,
        alpha=0.95,
    )

time_axis.axhline(0, color="#333333", linewidth=0.8)
time_axis.set_title("A. Quarterly out-of-sample performance", loc="left", fontweight="semibold")
time_axis.set_ylabel("Out-of-sample relative-valuation $R^2$")
time_axis.set_xlabel("Quarter")
time_axis.grid(alpha=0.22)
time_axis.legend(frameon=False, ncol=2, fontsize=9)

mean_values = (
    summary.set_index("model")
    .reindex(plot_models)["mean_test_variance_r2"]
)
positions = np.arange(len(plot_models))
bars = average_axis.bar(
    positions,
    mean_values,
    color=[colors[model] for model in plot_models],
    width=0.72,
)
average_axis.axhline(0, color="#333333", linewidth=0.8)
average_axis.set_xticks(positions, [plot_labels[model] for model in plot_models], rotation=35, ha="right")
average_axis.set_title("B. Mean across 80 quarters", loc="left", fontweight="semibold")
average_axis.grid(axis="y", alpha=0.22)
for bar, value in zip(bars, mean_values):
    average_axis.text(
        bar.get_x() + bar.get_width() / 2,
        value + (0.012 if value >= 0 else -0.012),
        f"{value:.1%}",
        ha="center",
        va="bottom" if value >= 0 else "top",
        fontsize=9,
    )

figure.suptitle("Full 80-Quarter Relative-Valuation Replication", fontweight="bold")
figure.tight_layout(rect=[0, 0, 1, 0.95])
for extension in ["png", "pdf", "svg"]:
    figure.savefig(VALIDATION_ROOT / "figures" / f"eighty_quarter_relative_valuation.{extension}", dpi=300, bbox_inches="tight")
plt.show()

print("Full 80-quarter replication pipeline completed.")
print(f"All validation outputs are stored under: {VALIDATION_ROOT}")


## Full-replication completion criteria

The full replication is considered complete when:

- all 80 quarters finish without manual data edits;
- every raw and derived checkpoint can be reloaded after a kernel restart;
- historical CUSIP match rates and sample sizes are economically plausible;
- minimum investor and firm degrees equal 20 after convergence;
- Gaussian placebo performance is approximately zero;
- Beta has low relative-valuation explanatory power;
- the four characteristics have positive explanatory power; and
- holdings embeddings materially outperform the characteristics in most validation quarters.

The saved quarterly outputs can subsequently feed the remaining recommender-system, Word2Vec, BERT, return-comovement, and masked-portfolio extensions.


## 13. Complete recommender-system matrices

The paper estimates three additional recommender-system variants:

- **RS-Ranks:** positive holdings are converted to within-investor percentile ranks. The largest position equals one, the smallest positive position equals (1/N_i), and nonpositions equal zero.
- **RS-L-0:** investor and asset fixed effects are removed from positive log dollar holdings, and nonpositions are assigned zero residual.
- **RS-L:** the same fixed-effect-adjusted positive log holdings are used, but nonpositions are ignored rather than imputed.

This section constructs and checkpoints the rank matrix and the fixed-effect-adjusted observed residuals. It uses only the finalized 80-quarter holdings already stored locally.


In [ ]:
RS_MODELS_ROOT = VALIDATION_ROOT / "additional_recommender_systems"
RS_MODELS_ROOT.mkdir(parents=True, exist_ok=True)

RS_DIMENSIONS = [4, 10]
ALS_REGULARIZATION_GRID = [0.1, 1.0, 10.0, 100.0, 1000.0]
ALS_MAX_ITERATIONS = 50
ALS_CV_ITERATIONS = 8
ALS_TOLERANCE = 1e-5
ALS_RANDOM_SEED = RANDOM_SEED + 3000
RS_L_MODEL_VERSION = 2
ALS_CV_FOLDS = 10


def construct_additional_rs_matrices(year, quarter):
    label = quarter_label(year, quarter)
    directory = RS_MODELS_ROOT / "matrices" / label
    rank_path = directory / "ranks.npz"
    residual_path = directory / "observed_log_residuals.npz"
    fixed_effect_path = directory / "fixed_effects.parquet"
    summary_path = directory / "matrix_summary.parquet"

    if (
        rank_path.exists()
        and residual_path.exists()
        and parquet_is_valid(summary_path, ["pipeline_version"])
        and not FORCE_REBUILD
    ):
        return sparse.load_npz(rank_path), sparse.load_npz(residual_path), pd.read_parquet(summary_path)

    holdings = pd.read_parquet(
        VALIDATION_ROOT / "processed" / label / "combined_final_holdings.parquet"
    )
    investor_mapping = pd.read_parquet(
        VALIDATION_ROOT / "matrices" / label / "investor_mapping.parquet"
    )
    firm_mapping = pd.read_parquet(
        VALIDATION_ROOT / "matrices" / label / "firm_mapping.parquet"
    )

    indexed = (
        holdings
        .merge(investor_mapping[["investor_key", "investor_index"]], on="investor_key", validate="many_to_one")
        .merge(firm_mapping[["permco", "firm_index"]], on="permco", validate="many_to_one")
    )
    indexed["investor_positions"] = indexed.groupby("investor_key")["permco"].transform("size")
    indexed["rank_value"] = (
        indexed["investor_positions"] - indexed["firm_rank"] + 1
    ) / indexed["investor_positions"]

    rows = indexed["investor_index"].to_numpy(dtype="int32")
    columns = indexed["firm_index"].to_numpy(dtype="int32")
    shape = (len(investor_mapping), len(firm_mapping))

    rank_matrix = sparse.csr_matrix(
        (indexed["rank_value"].to_numpy(dtype="float32"), (rows, columns)),
        shape=shape,
    )

    log_values = indexed["log_dollar_holding"].to_numpy(dtype="float64")
    investor_counts = np.bincount(rows, minlength=shape[0])
    investor_sums = np.bincount(rows, weights=log_values, minlength=shape[0])
    investor_fixed_effect = investor_sums / investor_counts
    investor_demeaned = log_values - investor_fixed_effect[rows]

    firm_counts = np.bincount(columns, minlength=shape[1])
    firm_sums = np.bincount(columns, weights=investor_demeaned, minlength=shape[1])
    firm_fixed_effect = firm_sums / firm_counts
    observed_residual = investor_demeaned - firm_fixed_effect[columns]

    residual_matrix = sparse.csr_matrix(
        (observed_residual.astype("float32"), (rows, columns)),
        shape=shape,
    )

    directory.mkdir(parents=True, exist_ok=True)
    sparse.save_npz(rank_path, rank_matrix, compressed=True)
    sparse.save_npz(residual_path, residual_matrix, compressed=True)

    fixed_effects = pd.DataFrame({
        "effect_type": ["investor"] * shape[0] + ["firm"] * shape[1],
        "index": np.concatenate([np.arange(shape[0]), np.arange(shape[1])]),
        "fixed_effect": np.concatenate([investor_fixed_effect, firm_fixed_effect]),
    })
    save_parquet(fixed_effects, fixed_effect_path)

    summary = pd.DataFrame({
        "year": [year],
        "quarter": [quarter],
        "investors": [shape[0]],
        "firms": [shape[1]],
        "positive_positions": [len(indexed)],
        "minimum_rank_value": [float(indexed["rank_value"].min())],
        "maximum_rank_value": [float(indexed["rank_value"].max())],
        "observed_residual_mean": [float(observed_residual.mean())],
        "observed_residual_std": [float(observed_residual.std(ddof=0))],
        "pipeline_version": [PIPELINE_VERSION],
    })
    save_parquet(summary, summary_path)
    return rank_matrix, residual_matrix, summary


additional_matrix_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    ranks, residuals, matrix_summary = construct_additional_rs_matrices(year, quarter)
    additional_matrix_inventory.append(matrix_summary)

additional_matrix_inventory = pd.concat(additional_matrix_inventory, ignore_index=True)
save_parquet(additional_matrix_inventory, RS_MODELS_ROOT / "matrices" / "eighty_quarter_matrix_summary.parquet")
display(additional_matrix_inventory.describe().T)


## 14. Estimate RS-Ranks and RS-L-0 by PCA

RS-Ranks and RS-L-0 are dense-matrix PCA models without regularization, exactly as described in Section 3.2.1 of the paper. RS-L-0 assigns zero to unobserved residual positions. We estimate up to 128 components once per quarter; the four- and ten-dimensional models use the leading coordinates from that common decomposition.


In [ ]:
def fit_pca_rs_model(year, quarter, model):
    if model not in {"rs_ranks", "rs_l_0"}:
        raise ValueError(model)

    label = quarter_label(year, quarter)
    directory = RS_MODELS_ROOT / "embeddings" / label / model
    embedding_path = directory / "asset_embeddings_normalized_labeled.parquet"
    fit_path = directory / "fit_summary.parquet"

    if (
        parquet_is_valid(embedding_path, ["permco", "embedding_001"])
        and parquet_is_valid(fit_path, ["pipeline_version"])
        and not FORCE_REBUILD
    ):
        return pd.read_parquet(embedding_path), pd.read_parquet(fit_path)

    rank_matrix, residual_matrix, _ = construct_additional_rs_matrices(year, quarter)
    matrix = rank_matrix if model == "rs_ranks" else residual_matrix
    dense = matrix.toarray().astype("float32")
    number_of_components = min(
        MAXIMUM_EMBEDDING_DIMENSION,
        dense.shape[0] - 1,
        dense.shape[1] - 1,
    )

    start = time.perf_counter()
    pca = PCA(
        n_components=number_of_components,
        svd_solver="randomized",
        random_state=RANDOM_SEED,
    ).fit(dense)

    asset_embeddings = normalized_asset_embeddings(pca, dense.shape[1])
    firm_mapping = pd.read_parquet(
        VALIDATION_ROOT / "matrices" / label / "firm_mapping.parquet"
    )
    labeled = firm_mapping[["permco", "firm_name", "firm_index"]].copy()
    embedding_frame = pd.DataFrame(
        asset_embeddings,
        columns=[f"embedding_{index + 1:03d}" for index in range(number_of_components)],
    )
    labeled = pd.concat([labeled.reset_index(drop=True), embedding_frame], axis=1)

    total_sum_squares = float(np.sum((dense - dense.mean(axis=0, keepdims=True)) ** 2))
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    fit_rows = []
    for dimension in [4, 10, 32, 64, 128]:
        if dimension <= number_of_components:
            fit_rows.append({
                "year": year,
                "quarter": quarter,
                "model": model,
                "dimension": dimension,
                "reconstruction_r2": float(cumulative[dimension - 1]),
                "investors": dense.shape[0],
                "firms": dense.shape[1],
                "fit_seconds": time.perf_counter() - start,
                "dense_matrix_mb": dense.nbytes / (1024 ** 2),
                "pipeline_version": PIPELINE_VERSION,
            })

    fit_summary = pd.DataFrame(fit_rows)
    directory.mkdir(parents=True, exist_ok=True)
    save_parquet(labeled, embedding_path)
    save_parquet(fit_summary, fit_path)
    np.save(directory / "explained_variance_ratio.npy", pca.explained_variance_ratio_)
    del dense
    return labeled, fit_summary


pca_rs_fit_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    for model in ["rs_ranks", "rs_l_0"]:
        _, fit_summary = fit_pca_rs_model(year, quarter, model)
        pca_rs_fit_inventory.append(fit_summary)
        print(f"{year}Q{quarter} {model}: complete")

pca_rs_fit_inventory = pd.concat(pca_rs_fit_inventory, ignore_index=True)
save_parquet(pca_rs_fit_inventory, RS_MODELS_ROOT / "embeddings" / "pca_rs_fit_summary.parquet")
display(pca_rs_fit_inventory.groupby(["model", "dimension"], as_index=False)["reconstruction_r2"].mean())


## 15. Estimate RS-L on positive positions by alternating least squares

RS-L uses only the intensive margin: its loss is evaluated exclusively on observed positive positions, and zero positions are ignored. Following Appendix B, we initialize from RS-L-0 and alternate ridge least-squares updates for investor and asset embeddings.

Regularization is selected separately for dimensions four and ten. A deterministic observed-position validation fold is used for model selection, after which the selected model is refitted on all positive positions. The fit is checkpointed separately for every quarter and dimension because RS-L dimensions are not nested PCA components.


In [ ]:
def als_update_factors(csr_values, left_factors, right_factors, regularization, update_left=True):
    source = csr_values if update_left else csr_values.T.tocsr()
    fixed = right_factors if update_left else left_factors
    updated = np.empty((source.shape[0], fixed.shape[1]), dtype="float64")
    identity = np.eye(fixed.shape[1], dtype="float64")
    for row_index in range(source.shape[0]):
        start, end = source.indptr[row_index], source.indptr[row_index + 1]
        indices = source.indices[start:end]
        values = source.data[start:end]
        design = fixed[indices]
        gram = design.T @ design + regularization * identity
        rhs = design.T @ values
        updated[row_index] = np.linalg.solve(gram, rhs)
    return updated


def observed_rmse(matrix, investor_factors, asset_factors):
    coo = matrix.tocoo()
    predictions = np.sum(investor_factors[coo.row] * asset_factors[coo.col], axis=1)
    return float(np.sqrt(np.mean((coo.data - predictions) ** 2)))


def fit_als(matrix, dimension, regularization, iterations, seed, initial_assets=None):
    rng = np.random.default_rng(seed)
    if initial_assets is None:
        asset_factors = rng.normal(scale=0.05, size=(matrix.shape[1], dimension))
    else:
        asset_factors = np.asarray(initial_assets[:, :dimension], dtype="float64").copy()
        scale = np.std(asset_factors)
        if scale > 0:
            asset_factors /= scale
    investor_factors = rng.normal(scale=0.05, size=(matrix.shape[0], dimension))
    history = []
    previous = None
    for iteration in range(1, iterations + 1):
        investor_factors = als_update_factors(matrix, investor_factors, asset_factors, regularization, True)
        asset_factors = als_update_factors(matrix, investor_factors, asset_factors, regularization, False)
        rmse = observed_rmse(matrix, investor_factors, asset_factors)
        history.append(rmse)
        if previous is not None and abs(previous - rmse) <= ALS_TOLERANCE * max(abs(previous), 1.0):
            break
        previous = rmse
    return investor_factors, asset_factors, history


def fit_rs_l(year, quarter, dimension):
    label = quarter_label(year, quarter)
    directory = RS_MODELS_ROOT / "embeddings" / label / "rs_l" / f"dimension={dimension}"
    embedding_path = directory / "asset_embeddings_labeled.parquet"
    fit_path = directory / "fit_summary.parquet"
    if parquet_is_valid(embedding_path, ["permco", "embedding_001"]) and parquet_is_valid(fit_path, ["pipeline_version", "rs_l_model_version"]) and not FORCE_REBUILD:
        return pd.read_parquet(embedding_path), pd.read_parquet(fit_path)

    _, residual_matrix, _ = construct_additional_rs_matrices(year, quarter)
    residual_matrix = residual_matrix.astype("float64").tocsr()
    coo = residual_matrix.tocoo()
    rng = np.random.default_rng(ALS_RANDOM_SEED + year * 10 + quarter + dimension)
    fold_assignments = rng.integers(0, ALS_CV_FOLDS, size=coo.nnz)

    l0_path = RS_MODELS_ROOT / "embeddings" / label / "rs_l_0" / "asset_embeddings_normalized_labeled.parquet"
    l0 = pd.read_parquet(l0_path)
    initial_columns = sorted(c for c in l0 if c.startswith("embedding_"))[:dimension]
    initial_assets = l0[initial_columns].to_numpy(dtype="float64")

    cv_rows = []
    for regularization in ALS_REGULARIZATION_GRID:
        for fold in range(ALS_CV_FOLDS):
            validation_mask = fold_assignments == fold
            training_matrix = sparse.csr_matrix(
                (coo.data[~validation_mask], (coo.row[~validation_mask], coo.col[~validation_mask])),
                shape=residual_matrix.shape,
            )
            investors_cv, assets_cv, history_cv = fit_als(
                training_matrix,
                dimension,
                regularization,
                ALS_CV_ITERATIONS,
                ALS_RANDOM_SEED + dimension + fold,
                initial_assets,
            )
            validation_predictions = np.sum(
                investors_cv[coo.row[validation_mask]] * assets_cv[coo.col[validation_mask]],
                axis=1,
            )
            validation_rmse = float(np.sqrt(np.mean((coo.data[validation_mask] - validation_predictions) ** 2)))
            cv_rows.append({
                "regularization": regularization,
                "fold": fold,
                "validation_rmse": validation_rmse,
                "iterations": len(history_cv),
            })

    cv_detail = pd.DataFrame(cv_rows)
    cv_summary = (
        cv_detail.groupby("regularization", as_index=False)
        .agg(
            validation_rmse=("validation_rmse", "mean"),
            validation_rmse_std=("validation_rmse", "std"),
            folds=("fold", "nunique"),
            mean_iterations=("iterations", "mean"),
        )
        .sort_values("validation_rmse")
        .reset_index(drop=True)
    )
    selected_regularization = float(cv_summary.iloc[0]["regularization"])
    start = time.perf_counter()
    investors, assets, history = fit_als(
        residual_matrix,
        dimension,
        selected_regularization,
        ALS_MAX_ITERATIONS,
        ALS_RANDOM_SEED + dimension,
        initial_assets,
    )

    # Resolve rotational scale for stable downstream standardization.
    covariance = assets.T @ assets / assets.shape[0]
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    whitening = eigenvectors @ np.diag(1.0 / np.sqrt(np.maximum(eigenvalues, 1e-10))) @ eigenvectors.T
    normalized_assets = (assets @ whitening).astype("float32")

    firm_mapping = pd.read_parquet(VALIDATION_ROOT / "matrices" / label / "firm_mapping.parquet")
    labeled = firm_mapping[["permco", "firm_name", "firm_index"]].copy()
    for index in range(dimension):
        labeled[f"embedding_{index + 1:03d}"] = normalized_assets[:, index]

    fit_summary = pd.DataFrame({
        "year": [year], "quarter": [quarter], "model": ["rs_l"], "dimension": [dimension],
        "selected_regularization": [selected_regularization],
        "observed_rmse": [observed_rmse(residual_matrix, investors, assets)],
        "iterations": [len(history)], "fit_seconds": [time.perf_counter() - start],
        "investors": [residual_matrix.shape[0]], "firms": [residual_matrix.shape[1]],
        "positive_positions": [residual_matrix.nnz], "pipeline_version": [PIPELINE_VERSION],
        "rs_l_model_version": [RS_L_MODEL_VERSION],
    })
    directory.mkdir(parents=True, exist_ok=True)
    save_parquet(labeled, embedding_path)
    save_parquet(fit_summary, fit_path)
    save_parquet(cv_summary, directory / "regularization_validation.parquet")
    save_parquet(cv_detail, directory / "regularization_validation_folds.parquet")
    return labeled, fit_summary


rs_l_fit_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    for dimension in RS_DIMENSIONS:
        _, fit_summary = fit_rs_l(year, quarter, dimension)
        rs_l_fit_inventory.append(fit_summary)
        print(f"{year}Q{quarter} RS-L {dimension}D: complete")

rs_l_fit_inventory = pd.concat(rs_l_fit_inventory, ignore_index=True)
save_parquet(rs_l_fit_inventory, RS_MODELS_ROOT / "embeddings" / "rs_l_fit_summary.parquet")
display(rs_l_fit_inventory.groupby("dimension", as_index=False).agg(mean_rmse=("observed_rmse", "mean"), mean_seconds=("fit_seconds", "mean"), max_iterations=("iterations", "max")))


## 16. Extend the paper-style valuation benchmark to all recommender systems

The new embeddings are evaluated on each quarter's already saved common firm sample and identical 80/20 split. This prevents changes in sample composition or random assignment from affecting comparisons with RS-Binary and RS-L-Min.

The final recommender-system table contains RS-Binary, RS-Ranks, RS-L-0, RS-L-Min, and RS-L at four and ten dimensions, along with the Gaussian placebo, Beta, and four-characteristic benchmarks.


In [ ]:
def evaluate_saved_embedding(year, quarter, model, dimension, embedding_frame):
    label = quarter_label(year, quarter)
    evaluation = pd.read_parquet(
        VALIDATION_ROOT / "benchmarks" / label / "evaluation_sample_and_split.parquet"
    )
    columns = sorted(c for c in embedding_frame if c.startswith("embedding_"))[:dimension]
    renamed = embedding_frame[["permco"] + columns].rename(columns={c: f"new_{c}" for c in columns})
    panel = evaluation.merge(renamed, on="permco", how="inner", validate="one_to_one")
    if len(panel) != len(evaluation):
        raise ValueError(f"{label} {model} {dimension}D does not cover the saved evaluation sample.")
    train = panel[panel["sample"] == "train"]
    test = panel[panel["sample"] == "test"]
    predictor_columns = [f"new_{c}" for c in columns]
    scaler = StandardScaler().fit(train[predictor_columns])
    X_train = scaler.transform(train[predictor_columns])
    X_test = scaler.transform(test[predictor_columns])
    y_train = train["relative_valuation"].to_numpy(dtype="float64")
    y_test = test["relative_valuation"].to_numpy(dtype="float64")
    cv = KFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
    ridge = RidgeCV(alphas=RIDGE_ALPHAS, cv=cv, scoring="neg_mean_squared_error").fit(X_train, y_train)
    prediction = ridge.predict(X_test)
    residual = y_test - prediction
    return {
        "year": year, "quarter": quarter, "model": f"{model}_{dimension}",
        "predictors": dimension, "firms": len(panel), "training_firms": len(train), "test_firms": len(test),
        "selected_alpha": float(ridge.alpha_),
        "test_variance_r2": float(1 - np.var(residual) / np.var(y_test)),
        "test_conventional_r2": float(r2_score(y_test, prediction)),
        "test_prediction_correlation": float(np.corrcoef(y_test, prediction)[0, 1]),
        "pipeline_version": PIPELINE_VERSION,
    }


additional_benchmark_rows = []
for year, quarter in VALIDATION_QUARTERS:
    label = quarter_label(year, quarter)
    for model in ["rs_ranks", "rs_l_0"]:
        embeddings = pd.read_parquet(RS_MODELS_ROOT / "embeddings" / label / model / "asset_embeddings_normalized_labeled.parquet")
        for dimension in RS_DIMENSIONS:
            additional_benchmark_rows.append(evaluate_saved_embedding(year, quarter, model, dimension, embeddings))
    for dimension in RS_DIMENSIONS:
        embeddings = pd.read_parquet(RS_MODELS_ROOT / "embeddings" / label / "rs_l" / f"dimension={dimension}" / "asset_embeddings_labeled.parquet")
        additional_benchmark_rows.append(evaluate_saved_embedding(year, quarter, "rs_l", dimension, embeddings))

additional_rs_benchmarks = pd.DataFrame(additional_benchmark_rows)
save_parquet(additional_rs_benchmarks, RS_MODELS_ROOT / "benchmarks" / "additional_rs_valuation_results.parquet")

existing_results = pd.read_parquet(VALIDATION_ROOT / "benchmarks" / "eighty_quarter_results.parquet")
existing_selection = existing_results[existing_results["model"].isin([
    "placebo_4", "placebo_10", "beta", "characteristics_4",
    "rs_binary_4", "rs_binary_10", "rs_l_min_4", "rs_l_min_10",
])].copy()
complete_rs_results = pd.concat([existing_selection, additional_rs_benchmarks], ignore_index=True, sort=False)
complete_rs_summary = (
    complete_rs_results.groupby("model", as_index=False)
    .agg(
        quarters=("quarter", "size"),
        mean_test_variance_r2=("test_variance_r2", "mean"),
        standard_deviation_test_r2=("test_variance_r2", "std"),
        minimum_test_variance_r2=("test_variance_r2", "min"),
        maximum_test_variance_r2=("test_variance_r2", "max"),
    )
    .sort_values("mean_test_variance_r2", ascending=False)
    .reset_index(drop=True)
)
save_parquet(complete_rs_results, RS_MODELS_ROOT / "benchmarks" / "complete_rs_valuation_results.parquet")
save_parquet(complete_rs_summary, RS_MODELS_ROOT / "benchmarks" / "complete_rs_valuation_summary.parquet")
display(complete_rs_summary)


## 17. Visualize the complete recommender-system comparison

This figure extends the paper-style valuation chart to all five recommender-system variants at dimensions four and ten. Word2Vec, PS-BERT, and OS-BERT will be appended to this same notebook after their project-local neural dependencies and checkpointed training routines are installed.


In [ ]:
model_order = [
    "placebo_4", "beta", "characteristics_4",
    "rs_binary_4", "rs_ranks_4", "rs_l_0_4", "rs_l_min_4", "rs_l_4",
    "rs_binary_10", "rs_ranks_10", "rs_l_0_10", "rs_l_min_10", "rs_l_10",
]
labels = {
    "placebo_4": "Placebo 4D", "beta": "Beta", "characteristics_4": "Four chars.",
    "rs_binary_4": "Binary 4D", "rs_ranks_4": "Ranks 4D", "rs_l_0_4": "L-0 4D", "rs_l_min_4": "L-Min 4D", "rs_l_4": "L 4D",
    "rs_binary_10": "Binary 10D", "rs_ranks_10": "Ranks 10D", "rs_l_0_10": "L-0 10D", "rs_l_min_10": "L-Min 10D", "rs_l_10": "L 10D",
}
plot_summary = complete_rs_summary.set_index("model").reindex(model_order)
if plot_summary["mean_test_variance_r2"].isna().any():
    raise ValueError("At least one complete recommender-system result is missing.")

colors = ["#AEB4BE", "#D2A62A", "#D9822B"] + ["#78A9D1"] * 5 + ["#286A9D"] * 5
figure, axis = plt.subplots(figsize=(16, 7))
positions = np.arange(len(model_order))
bars = axis.bar(positions, plot_summary["mean_test_variance_r2"], color=colors, width=0.72)
axis.axhline(0, color="#333333", linewidth=0.8)
axis.set_xticks(positions, [labels[model] for model in model_order], rotation=38, ha="right")
axis.set_ylabel("Mean quarterly out-of-sample relative-valuation $R^2$")
axis.set_title("Complete Recommender-System Benchmark, 2005Q1–2024Q4")
axis.grid(axis="y", alpha=0.22)
for bar, value in zip(bars, plot_summary["mean_test_variance_r2"]):
    axis.text(bar.get_x() + bar.get_width() / 2, value + 0.012, f"{value:.1%}", ha="center", va="bottom", fontsize=9)
figure.tight_layout()
figure_directory = VALIDATION_ROOT / "figures" / "complete_recommender_systems"
figure_directory.mkdir(parents=True, exist_ok=True)
for extension in ["png", "pdf", "svg"]:
    figure.savefig(figure_directory / f"complete_rs_relative_valuation.{extension}", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved complete recommender-system outputs under: {RS_MODELS_ROOT}")


## 18. Install the local Word2Vec training dependency

The Word2Vec reconstruction is trained locally and does not use the OpenAI API. PyTorch supplies automatic differentiation and Apple-silicon MPS acceleration when available. Installation is isolated to the active Jupyter environment. After a first installation, restart the kernel and resume at Section 19.


In [ ]:
%pip install torch


## 19. Configure the paper-faithful Word2Vec reconstruction

Following Section 3.3 and equations (5)–(6) of the paper, every investor portfolio remains ordered by decreasing portfolio weight. The model predicts each target firm from the average tied embedding of firms within a symmetric rank window, using full-vocabulary softmax cross-entropy. Context order is never randomized.

The paper does not disclose the context radius or optimizer settings. We therefore choose the context radius using masked-position validation loss on the same three diagnostic quarters used earlier, without consulting relative valuation. The selected radius is then fixed for all 80 production quarters.


In [ ]:
from src.paper_word2vec import (
    choose_device, portfolio_sequences, train_quarter,
)

WORD2VEC_ROOT = VALIDATION_ROOT / "word2vec"
WORD2VEC_ROOT.mkdir(parents=True, exist_ok=True)
WORD2VEC_DIMENSIONS = [4, 10]
WORD2VEC_RADIUS_CANDIDATES = [2, 5, 10]
WORD2VEC_CALIBRATION_QUARTERS = [(2005, 1), (2014, 4), (2024, 4)]
WORD2VEC_CONFIG = {
    "epochs": 8,
    "batch_size": 1024,
    "learning_rate": 0.01,
    "validation_fraction": 0.10,
    "seed": RANDOM_SEED + 6000,
}
WORD2VEC_CALIBRATION_CONFIG = {**WORD2VEC_CONFIG, "epochs": 3}
print(f"Word2Vec training device: {choose_device()}")


## 20. Select the undisclosed context radius on masked firms

This is an unsupervised calibration: the winning radius minimizes average held-out firm-prediction loss. Calibration artifacts are checkpointed, so rerunning the cell does not refit completed candidates.


In [ ]:
calibration_rows = []
for year, quarter in WORD2VEC_CALIBRATION_QUARTERS:
    label = quarter_label(year, quarter)
    sequences, firms = portfolio_sequences(
        VALIDATION_ROOT / "processed" / label / "combined_final_holdings.parquet",
        VALIDATION_ROOT / "matrices" / label / "investor_mapping.parquet",
        VALIDATION_ROOT / "matrices" / label / "firm_mapping.parquet",
    )
    for radius in WORD2VEC_RADIUS_CANDIDATES:
        _, result = train_quarter(
            sequences, firms, 4, radius,
            WORD2VEC_ROOT / "calibration" / label / f"radius={radius}",
            WORD2VEC_CALIBRATION_CONFIG, force=False,
        )
        row = result.iloc[0].to_dict()
        row.update({"year": year, "quarter": quarter})
        calibration_rows.append(row)
        print(f"{label}, radius {radius}: validation loss {row['best_validation_loss']:.4f}")

word2vec_calibration = pd.DataFrame(calibration_rows)
radius_summary = (
    word2vec_calibration.groupby("context_radius", as_index=False)
    .agg(mean_validation_loss=("best_validation_loss", "mean"), quarters=("quarter", "size"))
    .sort_values("mean_validation_loss")
    .reset_index(drop=True)
)
SELECTED_WORD2VEC_RADIUS = int(radius_summary.iloc[0]["context_radius"])
save_parquet(word2vec_calibration, WORD2VEC_ROOT / "calibration" / "quarter_results.parquet")
save_parquet(radius_summary, WORD2VEC_ROOT / "calibration" / "radius_summary.parquet")
display(radius_summary)
print(f"Selected context radius for the production run: {SELECTED_WORD2VEC_RADIUS}")


## 21. Train 4D and 10D Word2Vec embeddings for all 80 quarters

Each quarter and dimension has an independent checkpoint containing normalized firm embeddings, training history, validation loss, runtime, and device. Existing completed fits are reused after a kernel restart.


In [ ]:
radius_summary = pd.read_parquet(WORD2VEC_ROOT / "calibration" / "radius_summary.parquet")
SELECTED_WORD2VEC_RADIUS = int(radius_summary.iloc[0]["context_radius"])
word2vec_fit_rows = []
for year, quarter in VALIDATION_QUARTERS:
    label = quarter_label(year, quarter)
    sequences, firms = portfolio_sequences(
        VALIDATION_ROOT / "processed" / label / "combined_final_holdings.parquet",
        VALIDATION_ROOT / "matrices" / label / "investor_mapping.parquet",
        VALIDATION_ROOT / "matrices" / label / "firm_mapping.parquet",
    )
    for dimension in WORD2VEC_DIMENSIONS:
        _, result = train_quarter(
            sequences, firms, dimension, SELECTED_WORD2VEC_RADIUS,
            WORD2VEC_ROOT / "embeddings" / label / f"dimension={dimension}",
            WORD2VEC_CONFIG, force=False,
        )
        row = result.iloc[0].to_dict()
        row.update({"year": year, "quarter": quarter})
        word2vec_fit_rows.append(row)
        print(f"{label} Word2Vec {dimension}D: complete")
word2vec_fit_inventory = pd.DataFrame(word2vec_fit_rows)
save_parquet(word2vec_fit_inventory, WORD2VEC_ROOT / "embeddings" / "fit_inventory.parquet")
display(word2vec_fit_inventory.groupby("dimension", as_index=False).agg(
    quarters=("quarter", "size"), mean_validation_loss=("best_validation_loss", "mean"),
    mean_seconds=("fit_seconds", "mean"), maximum_epochs=("epochs_completed", "max"),
))


## 22. Add Word2Vec to the identical relative-valuation benchmark

The saved Word2Vec embeddings are merged into each quarter's existing evaluation panel. The sample, 80/20 firm split, ridge grid, ten-fold training cross-validation, and variance-based out-of-sample R-squared are unchanged, ensuring a clean comparison with the recommender systems and characteristics.


In [ ]:
word2vec_benchmark_rows = []
for year, quarter in VALIDATION_QUARTERS:
    label = quarter_label(year, quarter)
    for dimension in WORD2VEC_DIMENSIONS:
        embeddings = pd.read_parquet(
            WORD2VEC_ROOT / "embeddings" / label / f"dimension={dimension}" / "asset_embeddings_labeled.parquet"
        )
        word2vec_benchmark_rows.append(
            evaluate_saved_embedding(year, quarter, "word2vec", dimension, embeddings)
        )
word2vec_benchmarks = pd.DataFrame(word2vec_benchmark_rows)
save_parquet(word2vec_benchmarks, WORD2VEC_ROOT / "benchmarks" / "valuation_results.parquet")

complete_rs_results = pd.read_parquet(
    RS_MODELS_ROOT / "benchmarks" / "complete_rs_valuation_results.parquet"
)
all_model_results = pd.concat([complete_rs_results, word2vec_benchmarks], ignore_index=True, sort=False)
all_model_summary = (
    all_model_results.groupby("model", as_index=False)
    .agg(quarters=("quarter", "size"), mean_test_variance_r2=("test_variance_r2", "mean"),
         standard_deviation_test_r2=("test_variance_r2", "std"),
         minimum_test_variance_r2=("test_variance_r2", "min"),
         maximum_test_variance_r2=("test_variance_r2", "max"))
    .sort_values("mean_test_variance_r2", ascending=False).reset_index(drop=True)
)
save_parquet(all_model_results, WORD2VEC_ROOT / "benchmarks" / "all_models_results.parquet")
save_parquet(all_model_summary, WORD2VEC_ROOT / "benchmarks" / "all_models_summary.parquet")
display(all_model_summary)
